# Baseline Adult Income — exploration et choix du modele

**Objectif :** predire si le revenu annuel depasse 50K$ (`income` : 0 = `<=50K`, 1 = `>50K`).

Ce notebook sert a :
1. explorer le dataset
2. comparer quelques modeles
3. retenir la meilleure baseline a reporter dans `mlproject/train.py`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
plt.style.use("ggplot")
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline

from mlproject.config import TARGET
from mlproject.data import load_data, split
from mlproject.features import build_preprocessor

## 1. Chargement et exploration (EDA)

In [ ]:
df = load_data()
print("Shape:", df.shape)
df.head()

In [ ]:
print(df.dtypes)
print("\nValeurs manquantes:")
print(df.isna().sum().sort_values(ascending=False).head(10))

target_counts = df[TARGET].value_counts().sort_index()
print("\nDistribution de la cible:")
print(target_counts)
print(target_counts / len(df))

fig, ax = plt.subplots(figsize=(5, 4))
target_counts.plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"])
ax.set_title("Distribution income (0=<=50K, 1=>50K)")
ax.set_xlabel("income")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()

## 2. Comparaison de modeles

In [ ]:
X_train, X_test, y_train, y_test = split(df)

candidates = {
    "LogisticRegression": LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight="balanced",
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1,
    ),
}

results = []
fitted_models = {}

for name, clf in candidates.items():
    pipe = Pipeline([
        ("preprocessor", build_preprocessor()),
        ("clf", clf),
    ])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)
    results.append({
        "model": name,
        "f1": f1_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, proba),
    })
    fitted_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
results_df

## 3. Evaluation du modele retenu

On garde **LogisticRegression** (meilleur F1 sur ce dataset desequilibre).
Les memes hyperparametres sont reportes dans `mlproject/train.py`.

In [ ]:
BEST_MODEL_NAME = "LogisticRegression"
best_model = fitted_models[BEST_MODEL_NAME]

y_proba = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=["<=50K", ">50K"]))
print(f"f1={f1_score(y_test, y_pred):.3f}")
print(f"roc_auc={roc_auc_score(y_test, y_proba):.3f}")

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["<=50K", ">50K"]
)
plt.title(f"Matrice de confusion — {BEST_MODEL_NAME}")
plt.tight_layout()
plt.show()

### Hyperparametres retenus pour `train.py`

```python
LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced")
```

Resultats test (hold-out 20 %) :
- **f1 ≈ 0.677**
- **roc_auc ≈ 0.905**